FLOPs (Floating Point Operations) measure the computational complexity of neural network models by counting floating-point operations.

**MAC stands for Multiply-Accumulate Computation**.It measures a single combined hardware step that performs one multiplication followed immediately by one addition:

$$\text{Output} = \text{Output} + (a \times b)$$

$$\mathbf{2 \text{ MAC} = 1 \text{ FLOPs}}$$

This notebook evaluates **MACs (Multiply-Accumulate Operations)** and **Parameter Counts** for various Gemini/Gemma configurations.

1 Addition ($a + b$) = 1 FLOP

1 Subtraction ($a - b$) = 1 FLOP

1 Multiplication ($a \times b$) = 1 FLOP

1 Division ($a / b$) = 1 FLOP

In [3]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

import torch
from thop import profile
from Ch_4 import DummyGemini15FlashModel, BASE_CONFIG, GEMINI_MODEL_CONFIGS


In [4]:
BASE_CONFIG = {
    "vocab_size": 32000,     # Lowered from 256000 for CPU efficiency (prevents RAM hang)
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True,       
    "activation": "SwiGLU",  # Swish-Gated Linear Unit
}

# CPU-friendly lightweight Gemini/Google models
gemini_model_configs = {
    "gemini-flash-tiny": {"emb_dim": 256, "n_layers": 4, "n_heads": 8, "n_kv_heads": 8, "mlp_dim": 682},
    "gemini-nano-1": {"emb_dim": 512, "n_layers": 6, "n_heads": 8, "n_kv_heads": 4, "mlp_dim": 1408},
    "gemini-nano-2": {"emb_dim": 768, "n_layers": 8, "n_heads": 12, "n_kv_heads": 4, "mlp_dim": 2048},
    "gemini-1.5-flash": {"emb_dim": 3584, "n_layers": 42, "n_heads": 16, "n_kv_heads": 8, "mlp_dim": 14336},
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 2
#Creates a dummy tensor of shape (2, 1024) containing random token IDs between 0 and 255,999.
input_tensor = torch.randint(0, BASE_CONFIG["vocab_size"], (batch_size, 1024)).to(device)

for size in gemini_model_configs:
    cfg = BASE_CONFIG.copy()
    cfg.update(gemini_model_configs[size])

    model = DummyGemini15FlashModel(cfg).to(torch.bfloat16 if torch.cuda.is_available() else torch.float32)
    model.to(device)

    # MACS = multiply-accumulate operations
    macs, params = profile(model, inputs=(input_tensor,), verbose=False)
    flops = 2 * macs
    print(f"{size:18}: {flops:.1e} FLOPS | Params: {params/1e6:.2f}M")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

gemini-flash-tiny : 4.6e+10 FLOPS | Params: 11.34M
gemini-nano-1     : 1.4e+11 FLOPS | Params: 34.08M
gemini-nano-2     : 3.1e+11 FLOPS | Params: 74.92M
